# 🎓 تدريب «مخ البنش» على إجابات Claude

الدفتر ده بيعلّم النموذج نفسه (Qwen3.5-9B) من الأمثلة اللي جمعها البنش: إجابات Claude وتصحيحاته.
النتيجة ملف صغير اسمه **LoRA** (حوالي 50–150 ميجا) بتحطه جنب النموذج على الموبايل، فيتغيّر سلوك النموذج نفسه، مش بس اللي بنبعته له.

**قبل ما تبدأ:** من القايمة فوق **Runtime ← Change runtime type ← T4 GPU** ← Save.

شغّل الخلايا بالترتيب بزرار ▶ جنب كل خلية.

## 1) اتأكد إن فيه كارت شاشة

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2) سطّب الأدوات (بياخد 2–4 دقايق)

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade transformers trl datasets

## 3) ارفع ملف `mokh_train.jsonl`
اللي نزّلته من البنش: 🧠 ← «📦 نزّل بيانات تدريب المخ».

In [ ]:
from google.colab import files
up = files.upload()
DATA = next(iter(up))
import json
rows = [json.loads(l) for l in open(DATA, encoding='utf-8') if l.strip()]
print('عدد الأمثلة:', len(rows))
if len(rows) < 30:
    print('⚠️ الأمثلة قليلة. التدريب هيشتغل، بس الأحسن تجمع 50 مثال أو أكتر.')

## 4) حمّل النموذج (نفس النموذج اللي على الموبايل)

In [ ]:
from unsloth import FastModel
import torch
BASE = 'unsloth/Qwen3.5-9B'      # لازم يبقى نفس النموذج اللي على الموبايل
model, tok = FastModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=True)
model = FastModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=False, finetune_language_layers=True, random_state=7)
tok = getattr(tok, 'tokenizer', tok)

## 5) جهّز الأمثلة ودرّب

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
def fmt(r):
    return {'text': tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False)}
ds = Dataset.from_list(rows).map(fmt, remove_columns=list(rows[0].keys()))
steps = max(30, min(200, len(rows) * 3))
tr = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(dataset_text_field='text', per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   max_steps=steps, learning_rate=2e-4, warmup_steps=5, logging_steps=5,
                   optim='adamw_8bit', lr_scheduler_type='linear', seed=7, output_dir='out', report_to='none'))
tr.train()
model.save_pretrained('mokh_lora'); tok.save_pretrained('mokh_lora')
print('✔ التدريب خلص')

## 6) جرّبه هنا قبل ما تنزّله

In [ ]:
q = rows[0]['messages'][1]['content']
msgs = [rows[0]['messages'][0], {'role': 'user', 'content': q}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', enable_thinking=False).to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=300, temperature=0.5, do_sample=True)
print('السؤال:', q)
print('الرد:', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 7) حوّله لملف يشتغل على الموبايل ونزّله

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py mokh_lora --base-model-id Qwen/Qwen3.5-9B --outtype f16 --outfile mokh_lora.gguf
from google.colab import files
files.download('mokh_lora.gguf')

## 8) على الموبايل

1. انقل `mokh_lora.gguf` لـ Termux:
```
termux-setup-storage
cp ~/storage/downloads/mokh_lora.gguf ~/
```
2. شغّل المخ ومعاه الـ LoRA (بس زوّد `--lora`):
```
~/llama.cpp/build/bin/llama-server -m ~/qwen9b_q4.gguf --lora ~/mokh_lora.gguf -lm none -np 1 -c 4096 --host 127.0.0.1 --port 8080 --reasoning-budget 512
```
لو المخ اتلخبط أو بقى أسوأ، شغّله من غير `--lora` ترجع للنسخة الأصلية.